In [0]:
-- ============================================================================
-- 1. TOUR-LEVEL FEATURE SUMMARY (Your starting point)
-- ============================================================================

CREATE OR REPLACE TABLE production.supply_analytics.tour_feature_summary AS

WITH tour_option_features AS (
  SELECT 
    tour_id,
    tour_option_id,
    supplier_id,
    
    MAX(uses_prices_over_api) as uses_prices_over_api,
    MAX(uses_live_availability_and_price) as uses_live_availability_and_price,
    MAX(uses_pricing_scales_over_api) as uses_pricing_scales_over_api,
    MAX(uses_external_pricing) as uses_external_pricing,
    
    MAX(has_addons_configured) as has_addons_configured,
    MAX(has_individual_pricing) as has_individual_pricing,
    MAX(has_group_pricing) as has_group_pricing,
    MAX(has_any_scale_pricing) as has_any_scale_pricing,
    MAX(has_scale_pricing_individual) as has_scale_pricing_individual,
    MAX(has_scale_pricing_group) as has_scale_pricing_group,
    MAX(has_scale_pricing_addons) as has_scale_pricing_addons,
    
    MAX(has_cutoff_configured) as has_cutoff_configured,
    MAX(has_min_participant_requirement) as has_min_participant_requirement,
    MAX(has_max_participant_limit) as has_max_participant_limit,
    MAX(has_capacity_limits) as has_capacity_limits,
    
    AVG(num_individual_categories) as avg_num_individual_categories,
    AVG(num_group_categories) as avg_num_group_categories,
    AVG(num_addons) as avg_num_addons,
    AVG(total_individual_price_tiers) as avg_individual_tiers,
    AVG(total_group_price_tiers) as avg_group_tiers,
    AVG(total_addon_price_tiers) as avg_addon_tiers,
    AVG(total_price_tiers) as avg_total_price_tiers,
    
    AVG(pricing_dates_covered_next_12mo) as avg_dates_covered,
    AVG(pct_dates_with_vacancy_info) as avg_pct_vacancy_coverage
    
  FROM production.supply_analytics.feature_audit_base
  GROUP BY tour_id, tour_option_id, supplier_id
)

SELECT 
  t.tour_id,
  t.supplier_id,
  t.tour_title,
  t.location_name,
  t.activity_type,
  t.tour_category,
  t.date_of_creation,
  t.date_of_first_online,
  
  COUNT(DISTINCT tof.tour_option_id) as num_tour_options,
  
  MAX(tof.uses_prices_over_api) as uses_prices_over_api,
  MAX(tof.uses_live_availability_and_price) as uses_live_availability_and_price,
  MAX(tof.uses_pricing_scales_over_api) as uses_pricing_scales_over_api,
  MAX(tof.uses_external_pricing) as uses_external_pricing,
  
  MAX(tof.has_addons_configured) as has_addons_configured,
  MAX(tof.has_individual_pricing) as has_individual_pricing,
  MAX(tof.has_group_pricing) as has_group_pricing,
  MAX(tof.has_any_scale_pricing) as has_any_scale_pricing,
  MAX(tof.has_scale_pricing_individual) as has_scale_pricing_individual,
  MAX(tof.has_scale_pricing_group) as has_scale_pricing_group,
  MAX(tof.has_scale_pricing_addons) as has_scale_pricing_addons,
  
  MAX(tof.has_cutoff_configured) as has_cutoff_configured,
  MAX(tof.has_min_participant_requirement) as has_min_participant_requirement,
  MAX(tof.has_max_participant_limit) as has_max_participant_limit,
  MAX(tof.has_capacity_limits) as has_capacity_limits,
  
  ROUND(AVG(tof.avg_num_individual_categories), 1) as avg_num_individual_categories,
  ROUND(AVG(tof.avg_num_group_categories), 1) as avg_num_group_categories,
  ROUND(AVG(tof.avg_num_addons), 1) as avg_num_addons,
  
  ROUND(AVG(tof.avg_individual_tiers), 1) as avg_individual_price_tiers,
  ROUND(AVG(tof.avg_group_tiers), 1) as avg_group_price_tiers,
  ROUND(AVG(tof.avg_addon_tiers), 1) as avg_addon_price_tiers,
  ROUND(AVG(tof.avg_total_price_tiers), 1) as avg_total_price_tiers,
  
  ROUND(AVG(tof.avg_dates_covered), 0) as avg_dates_covered,
  ROUND(AVG(tof.avg_pct_vacancy_coverage), 1) as avg_pct_vacancy_coverage,
  
  (MAX(tof.has_addons_configured) + 
   MAX(tof.has_group_pricing) + 
   MAX(tof.has_any_scale_pricing) + 
   MAX(tof.has_cutoff_configured) + 
   MAX(tof.has_capacity_limits)) as tour_feature_count

FROM (
  SELECT DISTINCT 
    tour_id, supplier_id, tour_title, location_name, activity_type, 
    tour_category, date_of_creation, date_of_first_online
  FROM production.supply_analytics.feature_audit_base
) t
LEFT JOIN tour_option_features tof ON t.tour_id = tof.tour_id
GROUP BY t.tour_id, t.supplier_id, t.tour_title, t.location_name, t.activity_type, 
         t.tour_category, t.date_of_creation, t.date_of_first_online;


-- ============================================================================
-- 2. INDIVIDUAL PRICING CATEGORY DETAILS (EXPLODE JSON)
-- ============================================================================

CREATE OR REPLACE TABLE production.supply_analytics.individual_category_analysis AS

WITH exploded_individual AS (
  SELECT 
    tour_id,
    tour_option_id,
    supplier_id,
    location_name,
    activity_type,
    EXPLODE(individual_categories) as individual_cat
  FROM production.supply_analytics.feature_audit_base
  WHERE has_individual_pricing = 1
    AND uses_external_pricing = 0
)

SELECT 
  individual_cat.ticketCategory as ticket_category,
  individual_cat.minAge as min_age,
  individual_cat.maxAge as max_age,
  individual_cat.bookingType as booking_type,
  individual_cat.independentlyBookable as independently_bookable,
  SIZE(individual_cat.prices) as num_price_tiers,
  
  COUNT(DISTINCT tour_id) as num_tours_using,
  COUNT(*) as total_occurrences,
  
  ROUND(AVG(SIZE(individual_cat.prices)), 1) as avg_tiers_per_category,
  
  -- Most common locations using this category
  COLLECT_SET(location_name) as locations,
  COLLECT_SET(activity_type) as activity_types

FROM exploded_individual
GROUP BY 
  individual_cat.ticketCategory,
  individual_cat.minAge,
  individual_cat.maxAge,
  individual_cat.bookingType,
  individual_cat.independentlyBookable,
  SIZE(individual_cat.prices)
ORDER BY num_tours_using DESC;


-- ============================================================================
-- 3. INDIVIDUAL PRICING TIER STRUCTURE (WHAT DO TIERS LOOK LIKE?)
-- ============================================================================

CREATE OR REPLACE TABLE production.supply_analytics.individual_pricing_tier_patterns AS

WITH exploded_categories AS (
  SELECT 
    tour_id,
    tour_option_id,
    EXPLODE(individual_categories) as ind_cat
  FROM production.supply_analytics.feature_audit_base
  WHERE has_individual_pricing = 1
    AND uses_external_pricing = 0
),

exploded_prices AS (
  SELECT 
    tour_id,
    tour_option_id,
    ind_cat.ticketCategory as ticket_category,
    EXPLODE(ind_cat.prices) as price_tier
  FROM exploded_categories
)

SELECT 
  ticket_category,
  price_tier.maxParticipants as max_participants,
  
  COUNT(DISTINCT tour_id) as num_tours,
  COUNT(*) as occurrences,
  
  -- Price distribution
  ROUND(AVG(price_tier.retailPrice), 2) as avg_retail_price,
  ROUND(PERCENTILE(price_tier.retailPrice, 0.25), 2) as p25_retail_price,
  ROUND(PERCENTILE(price_tier.retailPrice, 0.50), 2) as median_retail_price,
  ROUND(PERCENTILE(price_tier.retailPrice, 0.75), 2) as p75_retail_price,
  ROUND(MIN(price_tier.retailPrice), 2) as min_retail_price,
  ROUND(MAX(price_tier.retailPrice), 2) as max_retail_price

FROM exploded_prices
GROUP BY ticket_category, price_tier.maxParticipants
ORDER BY ticket_category, max_participants;


-- ============================================================================
-- 4. GROUP PRICING ANALYSIS
-- ============================================================================

CREATE OR REPLACE TABLE production.supply_analytics.group_pricing_analysis AS

WITH exploded_group AS (
  SELECT 
    tour_id,
    tour_option_id,
    supplier_id,
    location_name,
    activity_type,
    EXPLODE(group_categories) as group_cat
  FROM production.supply_analytics.feature_audit_base
  WHERE has_group_pricing = 1
    AND uses_external_pricing = 0
)

SELECT 
  group_cat.isAdditionalPaxForGroup as is_additional_pax,
  SIZE(group_cat.prices) as num_price_tiers,
  
  COUNT(DISTINCT tour_id) as num_tours_using,
  COUNT(*) as total_occurrences,
  
  COLLECT_SET(location_name) as locations,
  COLLECT_SET(activity_type) as activity_types

FROM exploded_group
GROUP BY 
  group_cat.isAdditionalPaxForGroup,
  SIZE(group_cat.prices)
ORDER BY num_tours_using DESC;


-- ============================================================================
-- 5. GROUP PRICING TIER PATTERNS
-- ============================================================================

CREATE OR REPLACE TABLE production.supply_analytics.group_pricing_tier_patterns AS

WITH exploded_categories AS (
  SELECT 
    tour_id,
    tour_option_id,
    EXPLODE(group_categories) as grp_cat
  FROM production.supply_analytics.feature_audit_base
  WHERE has_group_pricing = 1
    AND uses_external_pricing = 0
),

exploded_prices AS (
  SELECT 
    tour_id,
    tour_option_id,
    grp_cat.isAdditionalPaxForGroup as is_additional_pax,
    EXPLODE(grp_cat.prices) as price_tier
  FROM exploded_categories
)

SELECT 
  is_additional_pax,
  price_tier.maxParticipants as max_participants,
  
  COUNT(DISTINCT tour_id) as num_tours,
  COUNT(*) as occurrences,
  
  ROUND(AVG(price_tier.retailPrice), 2) as avg_retail_price,
  ROUND(PERCENTILE(price_tier.retailPrice, 0.25), 2) as p25_retail_price,
  ROUND(PERCENTILE(price_tier.retailPrice, 0.50), 2) as median_retail_price,
  ROUND(PERCENTILE(price_tier.retailPrice, 0.75), 2) as p75_retail_price,
  ROUND(MIN(price_tier.retailPrice), 2) as min_retail_price,
  ROUND(MAX(price_tier.retailPrice), 2) as max_retail_price

FROM exploded_prices
GROUP BY is_additional_pax, price_tier.maxParticipants
ORDER BY is_additional_pax, max_participants;


-- ============================================================================
-- 6. ADDON ANALYSIS (WHAT ADDONS EXIST?)
-- ============================================================================

CREATE OR REPLACE TABLE production.supply_analytics.addon_analysis AS

WITH exploded_addons AS (
  SELECT 
    tour_id,
    tour_option_id,
    supplier_id,
    location_name,
    activity_type,
    EXPLODE(addons) as addon
  FROM production.supply_analytics.feature_audit_base
  WHERE has_addons_configured = 1
    AND uses_external_pricing = 0
)

SELECT 
  addon.addonId as addon_id,
  SIZE(addon.prices) as num_price_tiers,
  
  COUNT(DISTINCT tour_id) as num_tours_using,
  COUNT(*) as total_occurrences,
  
  ROUND(AVG(SIZE(addon.prices)), 1) as avg_tiers_per_addon,
  
  COLLECT_SET(location_name) as locations,
  COLLECT_SET(activity_type) as activity_types

FROM exploded_addons
GROUP BY 
  addon.addonId,
  SIZE(addon.prices)
ORDER BY num_tours_using DESC
LIMIT 100;


-- ============================================================================
-- 7. ADDON PRICING TIER PATTERNS
-- ============================================================================

CREATE OR REPLACE TABLE production.supply_analytics.addon_pricing_tier_patterns AS

WITH exploded_addons AS (
  SELECT 
    tour_id,
    tour_option_id,
    EXPLODE(addons) as addon
  FROM production.supply_analytics.feature_audit_base
  WHERE has_addons_configured = 1
    AND uses_external_pricing = 0
),

exploded_prices AS (
  SELECT 
    tour_id,
    tour_option_id,
    addon.addonId as addon_id,
    EXPLODE(addon.prices) as price_tier
  FROM exploded_addons
)

SELECT 
  price_tier.maxParticipants as max_participants,
  
  COUNT(DISTINCT tour_id) as num_tours,
  COUNT(DISTINCT addon_id) as num_unique_addons,
  COUNT(*) as occurrences,
  
  ROUND(AVG(price_tier.retailPrice), 2) as avg_retail_price,
  ROUND(PERCENTILE(price_tier.retailPrice, 0.25), 2) as p25_retail_price,
  ROUND(PERCENTILE(price_tier.retailPrice, 0.50), 2) as median_retail_price,
  ROUND(PERCENTILE(price_tier.retailPrice, 0.75), 2) as p75_retail_price,
  ROUND(MIN(price_tier.retailPrice), 2) as min_retail_price,
  ROUND(MAX(price_tier.retailPrice), 2) as max_retail_price

FROM exploded_prices
GROUP BY price_tier.maxParticipants
ORDER BY max_participants;


-- ============================================================================
-- 8. PRICING CATEGORY COMBINATION PATTERNS
-- ============================================================================

CREATE OR REPLACE TABLE production.supply_analytics.category_combination_patterns AS

WITH tour_categories AS (
  SELECT 
    tour_id,
    tour_option_id,
    COLLECT_SET(
      CASE 
        WHEN ind.ticketCategory IS NOT NULL 
        THEN CONCAT(ind.ticketCategory, '_', COALESCE(ind.minAge, 0), '-', COALESCE(ind.maxAge, 999))
      END
    ) as individual_category_set
  FROM production.supply_analytics.feature_audit_base
  LATERAL VIEW OUTER EXPLODE(individual_categories) exploded_ind AS ind
  WHERE uses_external_pricing = 0
  GROUP BY tour_id, tour_option_id
)

SELECT 
  SORT_ARRAY(individual_category_set) as category_combination,
  SIZE(individual_category_set) as num_categories_in_combo,
  
  COUNT(DISTINCT tour_id) as num_tours_using,
  COUNT(*) as occurrences

FROM tour_categories
WHERE SIZE(individual_category_set) > 0
GROUP BY individual_category_set
ORDER BY num_tours_using DESC
LIMIT 50;
